# Audio Recommendations

Loads precomputed BEATs embeddings and returns similar clips.

**Prereq:** run embedding creation once from a terminal (not this notebook):

```bash
python create_embeddings.py
```

That script is resumable — re-run after a crash and it continues from where it left off.

In [ ]:
!pip install pandas numpy scikit-learn faiss-cpu --quiet

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import faiss
    USE_FAISS = True
except ImportError:
    from sklearn.neighbors import NearestNeighbors
    USE_FAISS = False

CSV_PATH = Path("audio_speech_labels.csv")
EMBEDDINGS_PATH = Path("embeddings.npy")
ID_MAP_PATH = Path("id_map.json")
INDEX_PATH = Path("nn_index.faiss")

assert EMBEDDINGS_PATH.exists(), "Missing embeddings.npy — run: python create_embeddings.py"
assert ID_MAP_PATH.exists(), "Missing id_map.json — run: python create_embeddings.py"

all_embeddings = np.load(EMBEDDINGS_PATH)
with open(ID_MAP_PATH) as f:
    id_map = json.load(f)

df = pd.read_csv(CSV_PATH)
df = df[
    (df["speech_label"] == "non_speech")
    & (df["processing_status"] == "completed")
].reset_index(drop=True)

norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1, norms)
embeddings_normed = (all_embeddings / norms).astype(np.float32)

if USE_FAISS and INDEX_PATH.exists():
    index = faiss.read_index(str(INDEX_PATH))
    print(f"Loaded FAISS index: {index.ntotal} vectors")
elif USE_FAISS:
    index = faiss.IndexFlatIP(embeddings_normed.shape[1])
    index.add(embeddings_normed)
    faiss.write_index(index, str(INDEX_PATH))
    print(f"Built FAISS index: {index.ntotal} vectors")
else:
    index = NearestNeighbors(n_neighbors=11, metric="cosine", algorithm="brute")
    index.fit(embeddings_normed)
    print(f"sklearn NearestNeighbors fitted: {embeddings_normed.shape[0]} vectors")

id_to_idx = {cid: i for i, cid in enumerate(id_map)}
print(f"Ready: {len(id_map)} clips, dim={all_embeddings.shape[1]}")

In [ ]:
def recommend(clip_id: str, k: int = 10) -> pd.DataFrame:
    """Return top-k most similar clips for a given clip_id."""
    if clip_id not in id_to_idx:
        raise ValueError(f"clip_id '{clip_id}' not found in index.")

    idx = id_to_idx[clip_id]
    query_vec = embeddings_normed[idx : idx + 1]

    if USE_FAISS:
        scores, indices = index.search(query_vec, k + 1)
        scores, indices = scores[0], indices[0]
    else:
        distances, indices = index.kneighbors(query_vec, n_neighbors=k + 1)
        scores = 1 - distances[0]
        indices = indices[0]

    results = []
    for score, neighbor_idx in zip(scores, indices):
        neighbor_id = id_map[int(neighbor_idx)]
        if neighbor_id == clip_id:
            continue
        results.append({"id": neighbor_id, "similarity": float(score)})
        if len(results) >= k:
            break

    results_df = pd.DataFrame(results)
    meta_cols = [c for c in ["id", "duration", "speech_ratio", "total_duration_seconds", "streamableUrl"] if c in df.columns]
    return results_df.merge(df[meta_cols], on="id", how="left")

In [ ]:
# Sanity check: recommend for first 5 clips
for sample_id in id_map[:5]:
    print(f"\n{'='*60}")
    print(f"Query: {sample_id}")
    print("=" * 60)
    print(recommend(sample_id, k=5).to_string(index=False))

In [ ]:
# Query a specific clip
QUERY_ID = id_map[0]  # replace with any clip id from id_map
recommend(QUERY_ID, k=10)